---

# The story behind the project : 

## A restaurant manager wants to know how many people are needed, when they are needed, and how to schedule them fairly while controlling labor cost

---

---
---
---


In [48]:
import pandas as pd
import numpy as np
from pathlib import Path






In [49]:
PROJECT_ROOT = Path("..")
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"

print("Project root:", PROJECT_ROOT.resolve())
print("Raw data directory:", RAW_DATA_DIR.resolve())
print("Processed data directory:", PROCESSED_DATA_DIR.resolve())

Project root: /Users/homefolder/Desktop/it/Egna projekt/AI/machine-learning/restaurant-staff-scheduling-optimization
Raw data directory: /Users/homefolder/Desktop/it/Egna projekt/AI/machine-learning/restaurant-staff-scheduling-optimization/data/raw
Processed data directory: /Users/homefolder/Desktop/it/Egna projekt/AI/machine-learning/restaurant-staff-scheduling-optimization/data/processed


In [50]:
raw_csv_files = sorted(RAW_DATA_DIR.glob("*.csv"))

for file in raw_csv_files:
    print(file.name)

availability.csv
daily_staff_requirements.csv
menu_items.csv
orders.csv
staff.csv


In [51]:
# Load all raw CSV files into separate DataFrames

orders_df = pd.read_csv(RAW_DATA_DIR / "orders.csv")
menu_items_df = pd.read_csv(RAW_DATA_DIR / "menu_items.csv")
staff_df = pd.read_csv(RAW_DATA_DIR / "staff.csv")
availability_df = pd.read_csv(RAW_DATA_DIR / "availability.csv")
staff_requirements_df = pd.read_csv(RAW_DATA_DIR / "daily_staff_requirements.csv")

print("orders_df:", orders_df.shape)
print("menu_items_df:", menu_items_df.shape)
print("staff_df:", staff_df.shape)
print("availability_df:", availability_df.shape)
print("staff_requirements_df:", staff_requirements_df.shape)

orders_df: (157484, 25)
menu_items_df: (26, 9)
staff_df: (25, 11)
availability_df: (38983, 6)
staff_requirements_df: (2200, 9)


In [ ]:


dataset_overview = pd.DataFrame({
    "dataset": [
        "orders",
        "menu_items",
        "staff",
        "availability",
        "daily_staff_requirements"
    ],
    "rows": [
        orders_df.shape[0],
        menu_items_df.shape[0],
        staff_df.shape[0],
        availability_df.shape[0],
        staff_requirements_df.shape[0]
    ],
    "columns": [
        orders_df.shape[1],
        menu_items_df.shape[1],
        staff_df.shape[1],
        availability_df.shape[1],
        staff_requirements_df.shape[1]
    ],
    "missing_values": [
        orders_df.isna().sum().sum(),
        menu_items_df.isna().sum().sum(),
        staff_df.isna().sum().sum(),
        availability_df.isna().sum().sum(),
        staff_requirements_df.isna().sum().sum()
    ],
    "duplicate_rows": [
        orders_df.duplicated().sum(),
        menu_items_df.duplicated().sum(),
        staff_df.duplicated().sum(),
        availability_df.duplicated().sum(),
        staff_requirements_df.duplicated().sum()
    ]
})

dataset_overview

,dataset,rows,columns,missing_values,duplicate_rows
0,orders,157484,25,57050,314
1,menu_items,26,9,22,0
2,staff,25,11,21,0
3,availability,38983,6,315,90
4,daily_staff_requirements,2200,9,31,8


---

## Okej at the first look, we see that we have a big amount of missing values in the Orders data-set . Let's inspect it :

In [54]:
orders_missing_summary = (
    orders_df
    .isna()
    .sum()
    .reset_index()
)

orders_missing_summary.columns = ["column", "missing_count"]
orders_missing_summary["missing_percentage"] = (
    orders_missing_summary["missing_count"] / len(orders_df) * 100
).round(2)

orders_missing_summary = orders_missing_summary.sort_values(
    by="missing_count",
    ascending=False
)

orders_missing_summary

,column,missing_count,missing_percentage
10,table_number,56579,35.93
21,payment_method,124,0.08
13,menu_item_id,119,0.08
2,date,117,0.07
3,time,111,0.07
0,order_id,0,0.00
15,category,0,0.00
23,promotion_flag,0,0.00
22,cancelled,0,0.00
20,estimated_profit,0,0.00


## Let's inspect the

In [56]:
orders_structure_summary = pd.DataFrame({
    "column": orders_df.columns,
    "data_type": orders_df.dtypes.astype(str).values,
    "non_null_count": orders_df.notna().sum().values,
    "missing_count": orders_df.isna().sum().values,
    "missing_percentage": (orders_df.isna().sum().values / len(orders_df) * 100).round(2)
})

orders_structure_summary = orders_structure_summary.sort_values(
    by="missing_count",
    ascending=False
)

orders_structure_summary

,column,data_type,non_null_count,missing_count,missing_percentage
10,table_number,float64,100905,56579,35.93
21,payment_method,str,157360,124,0.08
13,menu_item_id,str,157365,119,0.08
2,date,str,157367,117,0.07
3,time,str,157373,111,0.07
0,order_id,str,157484,0,0.00
15,category,str,157484,0,0.00
23,promotion_flag,bool,157484,0,0.00
22,cancelled,str,157484,0,0.00
20,estimated_profit,float64,157484,0,0.00
